In [ ]:
using PyCall

py"""
import sys
sys.path.append("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils")
from solve_resonance import solve_resonance
"""

using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/Shannon_entropy.jl");

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_p*cos(ω_p*t))
    coupling = (cos((a+conj(a))/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (b+conj(b))/sqrt(2) * sqrt(Z_b)/ϕ_quantum)) / ϕ_quantum / sqrt(2)
    
        
    du[1] = -1im*(ω_a*a + sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b = 2*ω_a#7.1597*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_p=0.0

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_a = 0;#3e-3*κ_2 #from Leon's paper

In [ ]:
ϵ_p_list = 0.1:0.01:0.55pi
data = py"solve_resonance"(ϵ_p_list, ω_a, ω_b, φ_a, φ_b, E_J)
ω_d_vals = data["w_d"]
ω_p_vals = data["w_p"];

In [ ]:
t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 60/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save = dt
t_save = max(t_min_save,0):dt_save:t_max

a_0 = 0.0
b_0 = 0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

FFT_array_zero_freq_pump = Vector{Vector{Float64}}()
freqs_zero_freq_pump = Vector{Float64}()
ω_d_eff = ω_b
ϵ_p_list = 0.1:0.01:0.55pi
for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=20.12*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    u_0 = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    _, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)
    F, freqs_zero_freq_pump = FFT_perso(real.(b), t_list);
    push!(FFT_array_zero_freq_pump,abs.(F))
end
FFT_matrix_zero_freq_pump = reduce(hcat, FFT_array_zero_freq_pump);

In [ ]:
freqs_min, freqs_max = 0, 12
mask = (freqs_zero_freq_pump .>= freqs_min) .& (freqs_zero_freq_pump .<= freqs_max)
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b
heatmap(g2_list*1e3/2pi, freqs_zero_freq_pump[mask], log10.(FFT_matrix_zero_freq_pump[mask, :]), cmap = Perso_colormap)
plot!(g2_list*1e3/2pi, ω_d_vals/2/pi, ls=:dash, label="predicted "*L"\omega_d")
plot!(g2_list*1e3/2pi, abs.(ω_p_vals)/2/pi, ls=:dash, label="predicted "*L"\omega_p")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vacuum",
ylims = (freqs_min, freqs_max), xlims = (40,60),
colorbar_title = "\n"*L"\log_{10}|FFT|")

In [ ]:
ω_a = 3.8533*2*pi
ω_b = 7.1597*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

##Pump
ω_p=2ω_a-ω_b

t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 60/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save = dt
t_save = max(t_min_save,0):dt_save:t_max

a_0 = 0.0
b_0 = 0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

FFT_array = Vector{Vector{Float64}}()
freqs = Vector{Float64}()
ω_d_eff = ω_b
ϵ_p_list = 0.1:0.01:0.55pi
for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=20.12*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    u_0 = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    _, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)
    F, freqs = FFT_perso(real.(b), t_list);
    mask = (freqs .> 6) .& (freqs.<10)
    F_m = F[mask]
    freqs_m = freqs[mask]
    push!(FFT_array,abs.(F))
end
FFT_matrix = reduce(hcat, FFT_array);

In [ ]:
data = py"solve_resonance"(ϵ_p_list, ω_a, ω_b, φ_a, φ_b, E_J)
ω_d_vals = data["w_d"]
ω_p_vals = data["w_p"];

In [ ]:
g2_list[40]/2pi*1e3

In [ ]:
freqs_min, freqs_max = 0, 10
mask = (freqs .>= freqs_min) .& (freqs .<= freqs_max)
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b
heatmap(g2_list*1e3/2pi, freqs[mask], log10.(FFT_matrix[mask, :]), cmap = Perso_colormap)
#plot!(g2_list*1e3/2pi, ω_d_vals/2/pi, ls=:dash, label="predicted "*L"\omega_d")
#plot!(g2_list*1e3/2pi, abs.(ω_p_vals)/2/pi, ls=:dash, label="predicted "*L"\omega_p")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vacuum",
ylims = (freqs_min, freqs_max), xlims = (50, 55),
colorbar_title = "\n"*L"\log_{10}|FFT|")

In [ ]:
ϵ_p_list[72]

In [ ]:
freqs_min, freqs_max = 0, 10
mask = (freqs .>= freqs_min) .& (freqs .<= freqs_max)
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b
heatmap(g2_list*1e3/2pi, freqs[mask], log10.(FFT_matrix[mask, :]), cmap = Perso_colormap)
plot!(g2_list*1e3/2pi, ω_d_vals/2/pi, ls=:dash, label="predicted "*L"\omega_d")
plot!(g2_list*1e3/2pi, abs.(ω_p_vals)/2/pi, ls=:dash, label="predicted "*L"\omega_p")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vacuum",
ylims = (freqs_min, freqs_max), xlims = (5,40),
colorbar_title = "\n"*L"\log_{10}|FFT|")

# Sweeping the dissipation on the memory

In [ ]:
ω_a = 3.8533*2*pi
ω_b = 7.1597*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

##Pump
ω_p=2ω_a-ω_b

t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 60/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save = dt
t_save = max(t_min_save,0):dt_save:t_max

a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)

ϵ_p = .81
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=20.12*g2

drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

FFT_array = Vector{Vector{Float64}}()
freqs = Vector{Float64}()
ω_d_eff = ω_b
exp_κ_a_list = -9:0.05:3
κ_a_list = 10 .^exp_κ_a_list
for κ_a in κ_a_list
    environment = MeanField(κ_a, κ_b)
    environment_prepared = prepare(system,environment);
    u_0 = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    _, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)
    F, freqs = FFT_perso(real.(b), t_list);
    push!(FFT_array,abs.(F))
end
FFT_matrix = reduce(hcat, FFT_array);

In [ ]:
freqs_min, freqs_max = 0, 10
mask = (freqs .>= freqs_min) .& (freqs .<= freqs_max)
heatmap(exp_κ_a_list, freqs[mask], log10.(FFT_matrix[mask, :]), cmap = Perso_colormap)
plot!(xlabel=L"\log_{10}\kappa_a"*" (GHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vacuum",
ylims = (freqs_min, freqs_max),
colorbar_title = "\n"*L"\log_{10}|FFT|")

In [ ]:
FFT_array = Vector{Vector{Float64}}()
freqs = Vector{Float64}()
ω_d_eff = ω_b
exp_κ_a_list = -3.5:0.005:-2
κ_a_list = 10 .^exp_κ_a_list
for κ_a in κ_a_list
    environment = MeanField(κ_a, κ_b)
    environment_prepared = prepare(system,environment);
    u_0 = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    _, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)
    F, freqs = FFT_perso(real.(b), t_list);
    push!(FFT_array,abs.(F))
end
FFT_matrix = reduce(hcat, FFT_array);

In [ ]:
freqs_min, freqs_max = 0, 10
mask = (freqs .>= freqs_min) .& (freqs .<= freqs_max)
heatmap(exp_κ_a_list, freqs[mask], log10.(FFT_matrix[mask, :]), cmap = Perso_colormap)
plot!(xlabel=L"\log_{10}\kappa_a"*" (GHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vacuum",
ylims = (freqs_min, freqs_max),
colorbar_title = "\n"*L"\log_{10}|FFT|")

In [ ]:
ω_a = 3.8533*2*pi
ω_b = 7.1597*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

##Pump
ω_p=2ω_a-ω_b

t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 60/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save = dt
t_save = max(t_min_save,0):dt_save:t_max

a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)

ϵ_p = .81
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=20.12*g2

drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

FFT_array = Vector{Vector{Float64}}()
freqs = Vector{Float64}()
ω_d_eff = ω_b
exp_κ_a_list = -9:0.05:3
κ_a_list = 10 .^exp_κ_a_list
for κ_a in κ_a_list
    environment = MeanField(κ_a, κ_b)
    environment_prepared = prepare(system,environment);
    u_0 = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    _, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
    F, freqs = FFT_perso(real.(b), t_list);
    push!(FFT_array,abs.(F))
end
FFT_matrix = reduce(hcat, FFT_array);

In [ ]:
freqs_min, freqs_max = 0, 10
mask = (freqs .>= freqs_min) .& (freqs .<= freqs_max)
heatmap(exp_κ_a_list, freqs[mask], log10.(FFT_matrix[mask, :]), cmap = Perso_colormap)
plot!(xlabel=L"\log_{10}\kappa_a"*" (GHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vacuum Default",
ylims = (freqs_min, freqs_max),
colorbar_title = "\n"*L"\log_{10}|FFT|")